# ADS Bibliometric Analysis Pipeline

This notebook is the single entry point for the `ads_bib` package.
It executes all steps sequentially, with configuration cells between steps
so that downstream parameters can be set based on upstream results.

**Pipeline Steps:**
1. Search & Export â€” Query ADS API, resolve bibcodes to metadata
2. Translation â€” Detect languages, translate non-English titles/abstracts
3. Tokenization â€” Create full_text, tokenize with spaCy
4. AND â€” Optional external author name disambiguation
5. Topic Modeling & Curation â€” BERTopic + datamapplot + cluster removal
6. Citation Networks â€” Direct, Co-Citation, Bibliographic Coupling, Author Co-Citation


---
## Setup

In [ ]:
import time

from ads_bib.notebook import get_notebook_session

_pipeline_start = time.time()


In [ ]:
import logging

root_logger = logging.getLogger()
root_logger.setLevel(logging.INFO)
formatter = logging.Formatter("%(message)s")

if root_logger.handlers:
    for handler in root_logger.handlers:
        handler.setLevel(logging.INFO)
        handler.setFormatter(formatter)
else:
    handler = logging.StreamHandler()
    handler.setLevel(logging.INFO)
    handler.setFormatter(formatter)
    root_logger.addHandler(handler)

logger = logging.getLogger("pipeline")

---
## Global Run Control

Set run-level switches here. Phase-specific parameters are configured in each phase section below.


In [ ]:
# -- RUN CONFIGURATION --
# Set RESET_SESSION=True when you want a fresh run directory.
RESET_SESSION = False
RUN = {
    "run_name": "ADS_Curation_Run",
    "random_seed": 42,
    "openrouter_cost_mode": "hybrid",
}

session = get_notebook_session(
    run_name=RUN["run_name"],
    reset=RESET_SESSION,
    start_time=_pipeline_start,
)
session.set_section("run", RUN)


---
# Phase 1: Search & Export

Query the NASA ADS API for publications matching a search string,
then resolve all bibcodes (publications + references) into full metadata.

### 1.1 Search Configuration

Set query, years, and retrieval limits for your research question.
These values define the corpus scope for all downstream steps.


In [ ]:
# --- SEARCH CONFIGURATION ---
# Modify the query string for your research question.
# See: https://ui.adsabs.harvard.edu/help/search/search-syntax

set_a = "docs(library/P0hyiLw0T8qsyHSBpWigJA)"
set_b = f"citations({set_a}) AND year:1915-2000"
set_c = f"citations(citations({set_a})) AND year:1915-2000"
set_d = f"({set_a}) OR ({set_b}) OR ({set_c})"
set_t = "(docs(library/_-RCjJotRKCZC1PP03MqwA) AND year:1915-2000)"

gravity_relativity_terms = [
    "(general relativity)",
    "(gravitational field)",
    "(gravitation)",
    "(relativitatstheorie)",
    "(relativite generale)",
]
term_query = " OR ".join(gravity_relativity_terms)

SEARCH = {
    "query": f"({set_d}) AND ({term_query}) OR {set_t}",
    "ads_token": None,
    "refresh_search": True,
    "refresh_export": True,
}
session.set_section("search", SEARCH)


Alternative search ideas:

- Start from a smaller seed library and keep the same citation expansion pattern.
- Swap in multilingual term bundles for a narrower concept family.
- Update only `SEARCH["query"]`, then rerun the search/export cells.


### 1.2 Execute Search

Run the ADS search and persist bibcodes/results to this run folder.
This creates a reproducible input snapshot for later phases.


In [ ]:
session.run_stage("search")


### 1.3 Export & Resolve Metadata

Resolve publications and references to structured metadata tables.
This normalizes schemas before translation/tokenization/topic modeling.


In [ ]:
session.run_stage("export")


In [ ]:
if session.publications is not None:
    preview_cols = [
        c for c in ("Bibcode", "Year", "Author", "Title", "References")
        if c in session.publications.columns
    ]
    logger.info(
        "Phase 1 preview: publications=%s rows, columns=%s",
        f"{len(session.publications):,}",
        len(session.publications.columns),
    )
    if preview_cols:
        display(session.publications[preview_cols].head(10))
    else:
        display(session.publications.head(10))


---
# Phase 2: Translation

Detect non-English titles and abstracts with fasttext,
then translate them using either OpenRouter (API) or a local HuggingFace model.

### 2.1 Translation Configuration

Choose provider/model and translation target language.
Keep settings explicit so costs and outputs are reproducible.


In [ ]:
# -- TRANSLATION CONFIGURATION --
# Providers: openrouter, gguf, nllb
TRANSLATE = {
    "enabled": True,
    "provider": "openrouter",
    "model": "google/gemini-3-flash-preview",
    "api_key": None,
    "max_workers": 10,
    "max_tokens": 2048,
    "fasttext_model": str(session.paths["models"] / "lid.176.bin"),
}
session.set_section("translate", TRANSLATE)


### 2.2 Language Detection

Detect language tags for configured text columns.
Only non-target-language rows are translated in the next step.


In [ ]:
session.run_stage("translate")


### 2.3 Translate Non-English Entries

Translate non-English rows and track token/cost metadata.
This harmonizes text fields for downstream NLP and topic modeling.


In [ ]:
if session.publications is not None:
    preview_cols = [
        c for c in ("Title", "Title_en", "Abstract", "Abstract_en")
        if c in session.publications.columns
    ]
    if preview_cols:
        display(session.publications[preview_cols].head(5))


### 2.4 Save Translation Checkpoint

Persist translated data to global cache and run folder for Phase 3 restarts.

In [ ]:
logger.info(
    "Translated snapshot managed by shared runner under %s",
    session.paths["cache"],
)


---
# Phase 3: Tokenization

Create `full_text` (Title + Abstract) and tokenize publications with spaCy.
References are intentionally skipped in this phase for runtime stability.

In [ ]:
import os

# -- TOKENIZATION CONFIGURATION --
TOKENIZE = {
    "enabled": True,
    "spacy_model": "en_core_web_lg",
    "batch_size": 512,
    "n_process": min(max((os.cpu_count() or 1) - 1, 1), 8),
    "disable": ("ner", "parser", "textcat"),
    "fallback_model": "en_core_web_lg",
    "auto_download": True,
}
session.set_section("tokenize", TOKENIZE)


In [ ]:
session.run_stage("tokenize")


In [ ]:
if session.refs is not None:
    logger.info(
        "References dataset available at %s",
        session.run.paths["data"] / "references.parquet",
    )


### Phase 3 Snapshot

The shared runner manages tokenized publications and translated references under `data/cache/`.
Use the log cell below to confirm the current snapshot location.


In [ ]:
logger.info(
    "Tokenized snapshot managed by shared runner under %s",
    session.paths["cache"],
)


---
# Phase 4: Author Name Disambiguation

Optionally run an external AND package on the curated source datasets.
If disabled, the pipeline stores a Phase-4 checkpoint unchanged and continues.


In [ ]:
AUTHOR_DISAMBIGUATION = {
    "enabled": False,
    "model_bundle": None,
    "dataset_id": None,
    "force_refresh": False,
    "infer_stage": "full",
}
session.set_section("author_disambiguation", AUTHOR_DISAMBIGUATION)


In [ ]:
session.run_stage("author_disambiguation")


---
# Phase 5: Topic Modeling & Curation

Use BERTopic to discover thematic clusters, visualize with datamapplot,
then remove unwanted clusters to curate the dataset.

**Important:** Set parameters below based on your dataset size from Phase 1.

### 5.1 Embedding Configuration

Configure embedding provider/model and optional sampling.
These settings strongly affect topic quality and runtime/cost.

| Provider | Model Examples | Cost | Notes |
|----------|----------------|------|-------|
| `local` | `google/embeddinggemma-300m`, `BAAI/bge-large-en-v1.5` | None | CPU/GPU, no API needed |
| `huggingface_api` | `huggingface/BAAI/bge-large-en-v1.5` | Per-token | HF Inference API via LiteLLM |
| `openrouter` | `openai/text-embedding-3-large`, `google/gemini-embedding-001` | Per-token | Central billing, thread-pool concurrency |

**Caching**: Embeddings are cached to `data/cache/embeddings/` with SHA-256 fingerprint validation.
Changing the dataset or model triggers automatic recomputation.


In [ ]:
# -- EMBEDDING CONFIGURATION --
# Providers: openrouter, huggingface_api, local
TOPIC_MODEL = {
    "sample_size": None,
    "embedding_provider": "openrouter",
    "embedding_model": "google/gemini-embedding-001",
    "embedding_api_key": None,
    "embedding_batch_size": 64,
    "embedding_max_workers": 20,
}
session.set_section("topic_model", TOPIC_MODEL)


### 5.2 Compute Embeddings

Create or load cached embeddings for `full_text`.
Caching keeps repeated runs fast and reproducible.


In [ ]:
logger.info(
    "Topic stages reuse shared snapshots plus the embedding and reduction caches."
)


In [ ]:
session.run_stage("embeddings")


### 5.3 Dimensionality Reduction Configuration

Two projections are computed: **5D** (HDBSCAN clustering input) and **2D** (visualization & KDE).

| Method | Strengths | Weaknesses |
|--------|-----------|------------|
| `pacmap` | Fast, good local/global balance | No `densmap` (density-preserving mode) |
| `umap` | Supports `densmap=True` for KDE, better hierarchical structure | Slower, sensitive to `n_neighbors` |

**Key parameters:**

| Parameter | Default | Guidance |
|-----------|---------|----------|
| `n_neighbors` | 80 (PaCMAP) / 80 (UMAP) | Higher = more global structure; lower = more local detail |
| `min_dist` | 0.05 (UMAP only) | Lower = tighter clusters in 2D. Library default 0.1 is too loose for bibliometric data |
| `metric` | `angular`/`cosine` | PaCMAP uses `angular` (auto-converted from `cosine`) |
| `densmap` | `False` (UMAP only) | Set `True` in `PARAMS_2D` if you plan KDE density analysis downstream |

> **Tip**: If you plan KDE analysis on the 2D coordinates, use `method="umap"` with
> `PARAMS_2D = dict(..., densmap=True)`. Without densmap, UMAP inflates dense regions
> in the 2D projection, distorting density estimates.


In [ ]:
TOPIC_MODEL = {
    **TOPIC_MODEL,
    "reduction_method": "pacmap",
    "params_5d": {
        "n_neighbors": 80,
        "metric": "angular",
        "random_state": RUN["random_seed"],
    },
    "params_2d": {
        "n_neighbors": 80,
        "metric": "angular",
        "random_state": RUN["random_seed"],
    },
}
session.set_section("topic_model", TOPIC_MODEL)


In [ ]:
session.run_stage("reduction")


### 5.4 Clustering Configuration

HDBSCAN discovers topic clusters based on density in the 5D space.

| Parameter | Default | Guidance |
|-----------|---------|----------|
| `MIN_CLUSTER_SIZE` | `max(15, n_docs * 0.001)` | Controls granularity: ~0.1% of docs. Lower = more (smaller) topics |
| `min_samples` | 2–3 | Lower = fewer outliers but noisier clusters. Higher = stricter density |
| `cluster_selection_method` | `"eom"` | Excess of Mass: selects most persistent clusters |
| `cluster_selection_epsilon` | 0.02–0.05 | Absorbs border points; higher = larger clusters, fewer outliers |

**Backend choice:**
- `fast_hdbscan`: Fastest, but no `prediction_data` or `gen_min_span_tree` (no `approximate_predict()`).
- `hdbscan`: Supports `prediction_data=True` for predicting new documents and `gen_min_span_tree=True` for hierarchy analysis.

`BASE_MIN_CLUSTER_SIZE` is only used by Toponymy for micro-cluster granularity.


In [ ]:
TOPIC_MODEL = {
    **TOPIC_MODEL,
    "clustering_method": "fast_hdbscan",
    "cluster_params": {},
    "toponymy_cluster_params": {},
    "toponymy_evoc_cluster_params": {},
    "toponymy_layer_index": 1,
}
session.set_section("topic_model", TOPIC_MODEL)


### 5.5 Backend & LLM Configuration

**Backend matrix:**

| Backend | Clustering Input | LLM Providers | Notes |
|---------|-----------------|---------------|-------|
| `bertopic` | 5D reduced vectors | `local`, `gguf`, `huggingface_api`, `openrouter` | Standard BERTopic + outlier reduction |
| `toponymy` | 5D reduced vectors | `local`, `gguf`, `openrouter` | Hierarchical layers, richer labeling |
| `toponymy_evoc` | Raw embeddings | `local`, `gguf`, `openrouter` | EVoC clusterer, no 5D needed |

**Representation pipeline** (BERTopic):

| Model | Role | Configurable |
|-------|------|--------------|
| `POS` | Part-of-speech filtering (keep nouns, adjectives) | `pos_spacy_model` |
| `KeyBERT` | Semantic keyword re-ranking | Requires local embedding model |
| `MMR` | Maximal Marginal Relevance (diversity) | `mmr_diversity` (0–1) |
| LLM | Final topic label generation | Provider, model, prompt |

Default pipeline: **POS → KeyBERT → MMR → LLM** (sequential). Parallel models
stored separately in `topic_aspects_` for comparison.

`MIN_DF` guidance: `max(1, min(5, n_docs // 100))`. Small samples need `min_df=1`;
large corpora benefit from higher values to suppress noise terms.


### 5.5a LLM Prompt for Topic Labels

Choose a named prompt via `TOPIC_MODEL["llm_prompt_name"]` or provide a full override via
`TOPIC_MODEL["llm_prompt"]`.

Available prompt names:

- `physics`: gravitational physics, astrophysics, cosmology
- `generic`: domain-agnostic scientific labeling


In [ ]:
TOPIC_MODEL = {
    **TOPIC_MODEL,
    "llm_prompt_name": "physics",
    "llm_prompt": None,
}
session.set_section("topic_model", TOPIC_MODEL)


In [ ]:
TOPIC_MODEL = {
    **TOPIC_MODEL,
    "backend": "bertopic",
    "llm_provider": "openrouter",
    "llm_model": "google/gemini-3-flash-preview",
    "llm_api_key": None,
    "bertopic_label_max_tokens": 128,
    "toponymy_local_label_max_tokens": 256,
    "pipeline_models": ["POS", "KeyBERT", "MMR"],
    "parallel_models": ["MMR", "POS", "KeyBERT"],
    "toponymy_embedding_model": None,
    "toponymy_max_workers": 10,
    "min_df": None,
    "outlier_threshold": 0.5,
}
session.set_section("topic_model", TOPIC_MODEL)


### 5.5b Fit Topic Model

Run the selected backend (BERTopic or Toponymy) on reduced vectors.
This is the most compute/cost-intensive step of the pipeline.

In [ ]:
session.run_stage("topic_fit")


In [ ]:
if session.topic_info is not None:
    display(session.topic_info)


### 5.5c Build Topic DataFrame

Merge topic assignments, 2D coordinates, and embeddings into the main DataFrame.

In [ ]:
session.run_stage("topic_dataframe")


In [ ]:
if session.topic_df is not None:
    display(session.topic_df.head(10))


### 5.6 Visualize Topics

Render an interactive topic map from 2D coordinates and topic labels.
Use this view to inspect cluster semantics before curation.


In [ ]:
VISUALIZATION = {
    "enabled": True,
    "title": "ADS Bibliometric Map",
    "subtitle_template": "Topics labeled with {provider}/{model}",
    "dark_mode": True,
}
session.set_section("visualization", VISUALIZATION)


In [ ]:
session.run_stage("visualize")


### 5.7 Curate Dataset

Review the cluster summary, then remove clusters that don't fit your research question.

In [ ]:
CURATION = {
    "clusters_to_remove": [],
}
session.set_section("curation", CURATION)


In [ ]:
session.run_stage("curate")


### Curated Dataset Artifact

The curate stage writes the curated dataset to the current run directory.
Use the log cell below to confirm the handoff artifact for citation analysis.


In [ ]:
logger.info(
    "Curated dataset artifact: %s",
    session.run.paths["data"] / "publications.parquet",
)


In [ ]:
if session.curated_df is not None:
    from ads_bib.curate import get_cluster_summary

    display(get_cluster_summary(session.curated_df))
    display(session.curated_df.head(10))


---
# Phase 6: Citation Networks

Compute citation networks from the curated dataset and export
to GEXF (Gephi), Graphology JSON (Sigma.js), CSV, and/or Web of Science format.

### 6.1 Citation Configuration

Set network metrics, thresholds, filters, and export formats.
These parameters define which citation structures are constructed.

In [ ]:
CITATIONS = {
    "metrics": [
        "direct",
        "co_citation",
        "bibliographic_coupling",
        "author_co_citation",
    ],
    "min_counts": {
        "direct": 5,
        "co_citation": 20,
        "bibliographic_coupling": 20,
        "author_co_citation": 10,
    },
    "authors_filter": None,
    "output_format": "gexf",
}
session.set_section("citations", CITATIONS)


### 6.2 Build Node Table & Compute Networks

Build node metadata and compute selected citation networks.
Outputs are exported for Gephi/Sigma/CSV workflows.


In [ ]:
session.run_stage("citations")


### 6.3 Export Web of Science Format

Export the curated corpus in WOS text format.
This supports downstream tooling such as CiteSpace/VOSviewer.


In [ ]:
suffix = "_filtered" if session.config.citations.authors_filter else ""
logger.info(
    "WOS export artifact: %s",
    session.run.paths["data"] / f"download_wos_export{suffix}.txt",
)


---
# Summary

Final dataset statistics and output file listing.

In [ ]:
logger.info("Run artifacts: %s", session.run.paths["root"])


In [ ]:
session.save_summary()
